# Image Captioning with ViT and Transformer Decoder


## 1. Utils Module


In [ ]:
!pip install rouge-score
!pip install pycocoevalcap
!pip install transformers


In [ ]:
import nltk
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image
import os
import math

nltk.download('wordnet')


In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image

def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    step = checkpoint["step"]
    return step

def print_examples(model, device, dataset, num_samples=3):
    from transformers import ViTImageProcessor
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

    model.eval()
    seen_imgs = set()
    samples = []
    for img_id, caption in zip(dataset.imgs, dataset.captions):
        if img_id not in seen_imgs:
            seen_imgs.add(img_id)
            samples.append((img_id, caption))
        if len(samples) == num_samples:
            break

    for i, (img_id, correct_caption) in enumerate(samples):
        img_path = os.path.join(dataset.root_dir, img_id)
        img = Image.open(img_path).convert("RGB")
        test_img = processor(images=img, return_tensors="pt")['pixel_values'].to(device)

        res = model.caption_image(test_img, dataset.vocab)
        if isinstance(res, tuple):
            predicted_tokens, _ = res
        else:
            predicted_tokens = res

        predicted_tokens = [t for t in predicted_tokens if t not in ["<SOS>", "<EOS>", "<PAD>"]]

        try:
            res_beam = model.caption_image_beam_search(test_img, dataset.vocab, beam_size=3)
            if isinstance(res_beam, tuple):
                predicted_beam, _ = res_beam
            else:
                predicted_beam = res_beam
            predicted_beam = [t for t in predicted_beam if t not in ["<SOS>", "<EOS>", "<PAD>"]]
            cleaned_beam = []
            for token in predicted_beam:
                if cleaned_beam and token == "." and cleaned_beam[-1] == ".":
                    break
                cleaned_beam.append(token)
            beam_output_str = " ".join(cleaned_beam)
        except AttributeError:
            beam_output_str = "(Mô hình hiện tại không hỗ trợ hoặc chưa định nghĩa hàm Beam Search)"

        cleaned = []
        for token in predicted_tokens:
            if cleaned and token == "." and cleaned[-1] == ".":
                break
            cleaned.append(token)

        print(f"  [Example {i+1}] CORRECT : {correct_caption}")
        print(f"  [Example {i+1}] OUTPUT (GREEDY)  : {' '.join(cleaned)}")
        print(f"  [Example {i+1}] OUTPUT (BEAM K=3)  : {beam_output_str}")
        print("-" * 40)
    model.train()


## 2. Dataset Module


In [ ]:
import pandas as pd
import spacy
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import ViTImageProcessor

spacy_eng = spacy.load("en_core_web_sm")

class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        return [tok.text.lower() for tok in spacy_eng.tokenizer(text)]

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                if word not in frequencies:
                    frequencies[word] = 1
                else:
                    frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

class FlickrDataset(Dataset):
    def __init__(self, root_dir, df, vocab=None, transform=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = df
        
        self.imgs = self.df['image_name'].tolist()
        self.captions = self.df['comment'].tolist()
        
        self.transform = transform

        if vocab is None:
            self.vocab = Vocabulary(freq_threshold)
            self.vocab.build_vocabulary(self.captions)
        else:
            self.vocab = vocab

        self.numericalized_captions = []
        for caption in self.captions:
            num_cap = [self.vocab.stoi["<SOS>"]]
            num_cap += self.vocab.numericalize(caption)
            num_cap.append(self.vocab.stoi["<EOS>"])
            self.numericalized_captions.append(torch.tensor(num_cap))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]
        img_path = os.path.join(self.root_dir, img_id)
        
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, self.numericalized_captions[index]

class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=False, padding_value=self.pad_idx)

        return imgs, targets

def get_loaders(
    root_folder,
    annotation_file,
    transform_train,
    transform_val_test,
    batch_size=32,
    num_workers=0,
    pin_memory=True,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42
):
    import random
    
    df = pd.read_csv(annotation_file)
    
    unique_images = df['image_name'].unique().tolist()
    random.seed(seed)
    random.shuffle(unique_images)
    
    total_imgs = len(unique_images)
    val_size = int(total_imgs * val_ratio)
    test_size = int(total_imgs * test_ratio)
    
    test_imgs = unique_images[:test_size]
    val_imgs = unique_images[test_size : test_size + val_size]
    train_imgs = unique_images[test_size + val_size:]
    
    train_df = df[df['image_name'].isin(train_imgs)]
    val_df = df[df['image_name'].isin(val_imgs)]
    test_df = df[df['image_name'].isin(test_imgs)]
    
    train_dataset = FlickrDataset(root_folder, train_df, transform=transform_train)
    val_dataset = FlickrDataset(root_folder, val_df, transform=transform_val_test, vocab=train_dataset.vocab)
    test_dataset = FlickrDataset(root_folder, test_df, transform=transform_val_test, vocab=train_dataset.vocab)

    pad_idx = train_dataset.vocab.stoi["<PAD>"]

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )
    
    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )
    
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=False,
        pin_memory=pin_memory,
        collate_fn=MyCollate(pad_idx=pad_idx),
    )

    return train_loader, val_loader, test_loader, train_dataset


## 2.5 Visualize Dataset Samples


In [16]:
def get_loader():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),         
        transforms.RandomHorizontalFlip(p=0.5),    
        transforms.ColorJitter(brightness=0.2),    
        transforms.ToTensor(),                     
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    train_loader, val_loader, test_loader, train_dataset = get_loaders(
        root_folder = "/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images",
        annotation_file = "/kaggle/input/datasets/eeshawn/flickr30k/captions.txt",
        transform_train = train_transform,
        transform_val_test = val_transform,
        num_workers=4,
    )
    return train_loader, val_loader, test_loader, train_dataset

In [17]:
train_loader, val_loader, test_loader, train_dataset = get_loader()

In [ ]:
def show_dataset_samples(dataset, num_samples=3):
    import matplotlib.pyplot as plt
    import random
    
    fig = plt.figure(figsize=(15, 5 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)
    
    for i, idx in enumerate(indices):
        img_id = dataset.imgs[idx]
        caption = dataset.captions[idx]
        img_path = os.path.join(dataset.root_dir, img_id)
        
        img = Image.open(img_path).convert("RGB")
        
        ax = fig.add_subplot(num_samples, 1, i+1)
        ax.imshow(img)
        ax.set_title(f"Caption: {caption}", fontsize=14, wrap=True)
        ax.axis("off")
        
    plt.tight_layout()
    plt.show()

show_dataset_samples(train_dataset, num_samples=3)


## 3. Model Module


In [ ]:
import torch.nn as nn
from transformers import ViTModel

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.3, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class EncoderViT(nn.Module):
    def __init__(self, embed_size, train_vit=False, num_trainable_blocks=2):
        super(EncoderViT, self).__init__()
        self.train_vit = train_vit
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        

        self.adapter = nn.Sequential(
            nn.Linear(768, embed_size),
            nn.LayerNorm(embed_size)
        )

        if not self.train_vit:
            for param in self.vit.parameters():
                param.requires_grad = False

        if self.train_vit:
            for param in self.vit.layernorm.parameters():
                param.requires_grad = True
                

            for block in self.vit.encoder.layer[-num_trainable_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

    def forward(self, images):
        features = self.vit(pixel_values=images).last_hidden_state
        features = self.adapter(features)
        return features

    def train(self, mode=True):
        super(EncoderViT, self).train(mode)
        if not self.train_vit:
            self.vit.eval()

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads=8, num_layers=3, dropout=0.3):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = PositionalEncoding(embed_size, dropout)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, dropout=dropout)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.linear = nn.Linear(embed_size, vocab_size)
        self.embed_size = embed_size

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, features, captions, pad_mask=None):

        memory = features.permute(1, 0, 2) 
        
        tgt = self.embed(captions) * math.sqrt(self.embed_size)
        tgt = self.pos_encoder(tgt)
        
        tgt_mask = self.generate_square_subsequent_mask(len(captions)).to(captions.device)
        
        output = self.transformer_decoder(tgt, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=pad_mask)
        output = self.linear(output)
        
        return output

class ViTtoTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads=8, num_layers=3, dropout=0.3, train_vit=False):
        super(ViTtoTransformer, self).__init__()
        self.encoderViT = EncoderViT(embed_size, train_vit)
        self.decoderTransformer = DecoderTransformer(embed_size, vocab_size, num_heads=num_heads, num_layers=num_layers, dropout=dropout)

    def forward(self, images, captions, pad_mask=None):
        features = self.encoderViT(images)
        outputs = self.decoderTransformer(features, captions[:-1], pad_mask)
        return outputs

    def caption_image(self, image, vocabulary, max_length=50):
        result_caption = []
        alphas = []
        device = image.device
        
        with torch.no_grad():
            features = self.encoderViT(image)
            memory = features.permute(1, 0, 2)
            
            tgt_indexes = [vocabulary.stoi["<SOS>"]]
            
            for _ in range(max_length):
                tgt_tensor = torch.LongTensor(tgt_indexes).unsqueeze(1).to(device) 
                
                tgt = self.decoderTransformer.embed(tgt_tensor) * math.sqrt(self.decoderTransformer.embed_size)
                tgt = self.decoderTransformer.pos_encoder(tgt)
                tgt_mask = self.decoderTransformer.generate_square_subsequent_mask(len(tgt_indexes)).to(device)
                
                output = self.decoderTransformer.transformer_decoder(tgt, memory, tgt_mask=tgt_mask)
                
                last_output = output[-1, 0, :].unsqueeze(0)
                mem_features = memory[:, 0, :]
                embed_size = self.decoderTransformer.embed_size
                attn_scores = torch.matmul(last_output, mem_features.transpose(0, 1)) / math.sqrt(embed_size)
                alpha = torch.softmax(attn_scores, dim=-1)
                alpha_image_patches = alpha.squeeze(0)[1:]
                alphas.append(alpha_image_patches.cpu())
                
                logits = self.decoderTransformer.linear(output) 
                pred_token = logits[-1, 0, :].argmax().item()
                
                result_caption.append(pred_token)
                tgt_indexes.append(pred_token)
                
                if vocabulary.itos[pred_token] == "<EOS>":
                    break
                    
            alphas_tensor = torch.stack(alphas) if len(alphas) > 0 else None
                    
        return [vocabulary.itos[idx] for idx in result_caption], alphas_tensor

    def caption_image_beam_search(self, image, vocabulary, beam_size=3, max_length=50):
        result_caption = []
        device = image.device
        
        with torch.no_grad():
            features = self.encoderViT(image)
            memory = features.permute(1, 0, 2)
            start_token = vocabulary.stoi["<SOS>"]
            beams = [(0.0, [start_token], [])]
            completed_beams = []
            
            for _ in range(max_length):
                new_beams = []
                for score, seq, alphas in beams:
                    if seq[-1] == vocabulary.stoi["<EOS>"]:
                        completed_beams.append((score, seq, alphas))
                        continue
                        
                    tgt_tensor = torch.LongTensor(seq).unsqueeze(1).to(device)
                    tgt = self.decoderTransformer.embed(tgt_tensor) * math.sqrt(self.decoderTransformer.embed_size)
                    tgt = self.decoderTransformer.pos_encoder(tgt)
                    tgt_mask = self.decoderTransformer.generate_square_subsequent_mask(len(seq)).to(device)
                    
                    output = self.decoderTransformer.transformer_decoder(tgt, memory, tgt_mask=tgt_mask)

                    last_output = output[-1, 0, :].unsqueeze(0)
                    mem_features = memory[:, 0, :]
                    embed_size = self.decoderTransformer.embed_size
                    attn_scores = torch.matmul(last_output, mem_features.transpose(0, 1)) / math.sqrt(embed_size)
                    alpha = torch.softmax(attn_scores, dim=-1)
                    alpha_patches = alpha.squeeze(0)[1:].cpu()
                
                    logits = self.decoderTransformer.linear(output)[-1, 0, :]
                    log_probs = torch.nn.functional.log_softmax(logits, dim=0)
                    
                    top_log_probs, top_idx = log_probs.topk(beam_size)
                    
                    for i in range(beam_size):
                        new_score = score + top_log_probs[i].item()
                        new_seq = seq + [top_idx[i].item()]
                        new_alphas = alphas + [alpha_patches]
                        new_beams.append((new_score, new_seq, new_alphas))
                
                if not new_beams:
                    break
                    
                new_beams.sort(key=lambda x: x[0], reverse=True)
                beams = new_beams[:beam_size]
                
            for score, seq, alphas in beams:
                if seq[-1] != vocabulary.stoi["<EOS>"]:
                    completed_beams.append((score, seq, alphas))
                    
            completed_beams.sort(key=lambda x: x[0], reverse=True)
            # best_seq = completed_beams[0][1]
            best_score, best_seq, best_alphas = completed_beams[0]
            
            result_caption = [vocabulary.itos[idx] for idx in best_seq]
            # alphas_tensor = torch.empty(0)
            alphas_tensor  = torch.stack(best_alphas) if best_alphas else torch.empty(0)
                
        return result_caption, alphas_tensor


## 4. Training Module


In [ ]:

import os
import torch
from collections import defaultdict
from tqdm import tqdm
from PIL import Image
from transformers import ViTImageProcessor

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

def evaluate_all_metrics(model, test_dataset, device, use_beam_search=False, beam_size=3):
    mode_name = f"Beam Search (k={beam_size})" if use_beam_search else "Greedy Search"
    print(f"Bắt đầu chuẩn bị dữ liệu đánh giá trên tập Test bằng phương pháp: {mode_name}...")
    model.eval()
    
    gts = defaultdict(list)
    for img_id, caption in zip(test_dataset.imgs, test_dataset.captions):
        clean_cap = caption.strip().lower()
        gts[img_id].append(clean_cap)
        
    res = {}
    
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    unique_imgs = list(gts.keys())
    
    with torch.no_grad():
        for img_id in tqdm(unique_imgs, desc=f"Generating Predictions ({mode_name})"):
            img_path = os.path.join(test_dataset.root_dir, img_id)
            img = Image.open(img_path).convert("RGB")
            
            pixel_values = processor(images=img, return_tensors="pt")['pixel_values'].to(device)
            
            if use_beam_search:
                output = model.caption_image_beam_search(pixel_values, test_dataset.vocab, beam_size=beam_size)
            else:
                output = model.caption_image(pixel_values, test_dataset.vocab)
                
            predicted_tokens = output[0] if isinstance(output, tuple) else output
            
            cleaned_pred = [t for t in predicted_tokens if t not in ["<SOS>", "<EOS>", "<PAD>"]]
            
            final_clean = []
            for token in cleaned_pred:
                if final_clean and token == "." and final_clean[-1] == ".":
                    break
                final_clean.append(token)
                
            pred_sentence = " ".join(final_clean).strip().lower()
            res[img_id] = [pred_sentence]

    print("\nĐang tính toán các chỉ số (BLEU, METEOR, ROUGE-L, CIDEr)...")
    
    scorers = [
        (Bleu(4), ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]),
        (Meteor(), "METEOR"),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]
    
    results = {}
    
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        
        if isinstance(method, list): 
            for m, s in zip(method, score):
                results[m] = s
                print(f"{m:10s}: {s * 100:.2f}")
        else:
            results[method] = score
            print(f"{method:10s}: {score * 100:.2f}")
            
    return results

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train():
    train_loader, val_loader, test_loader, train_dataset = get_loader()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    embed_size = 512
    vocab_size = 0 
    num_heads = 8
    num_layers = 3
    dropout = 0.3
    learning_rate = 3e-4
    num_epochs = 1
    batch_size = 64

    warmup_epochs = 5
    
    root_folder = "/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images"
    annotation_file = "/kaggle/input/datasets/eeshawn/flickr30k/captions.txt"
    
    
    if not os.path.exists(root_folder):
        print("Vui lòng cập nhật đường dẫn root_folder và annotation_file cho Kaggle.")
        return
        
    
    vocab_size = len(train_dataset.vocab)
    pad_idx = train_dataset.vocab.stoi["<PAD>"]
    
    model = ViTtoTransformer(
        embed_size=embed_size,
        vocab_size=vocab_size,
        num_heads=num_heads,
        num_layers=num_layers,
        dropout=dropout,
        train_vit=False
    ).to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    
    scaler = torch.amp.GradScaler('cuda')
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        if epoch == warmup_epochs:
            print("\n" + "="*60)
            print(f"[EPOCH {epoch+1}] CHUYỂN PHA: BẮT ĐẦU FINE-TUNING ViT ENCODER")
            print("="*60)
            
            model.encoderViT.train_vit = True
            
            for param in model.encoderViT.vit.layernorm.parameters():
                param.requires_grad = True
            for block in model.encoderViT.vit.encoder.layer[-2:]:
                for param in block.parameters():
                    param.requires_grad = True
                    
            vit_params = []
            decoder_adapter_params = []
            for name, param in model.named_parameters():
                if not param.requires_grad: continue
                if "encoderViT.vit" in name:
                    vit_params.append(param)
                else:
                    decoder_adapter_params.append(param)
                    

            optimizer.param_groups[0]['lr'] = 1e-4 
            optimizer.add_param_group({"params": vit_params, "lr": 1e-5})
            
            print("--> Đã nạp Optimizer & Scheduler mới thành công!\n")
            
        model.train()
        loop = tqdm(train_loader, total=len(train_loader), leave=True)
        
        for idx, (imgs, captions) in enumerate(loop):

            imgs = imgs.to(device)
            captions = captions.to(device)
            
            pad_mask = (captions[:-1] == pad_idx).transpose(0, 1)
            targets = captions[1:]
            
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(imgs, captions, pad_mask=pad_mask)
                loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
            loop.set_postfix(loss=loss.item())
            
        # Validation Loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, captions in val_loader:
                imgs = imgs.to(device)
                captions = captions.to(device)
                pad_mask = (captions[:-1] == pad_idx).transpose(0, 1)
                outputs = model(imgs, captions, pad_mask=pad_mask)
                targets = captions[1:]

                with torch.amp.autocast('cuda'):
                    outputs = model(imgs, captions, pad_mask=pad_mask)
                    loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                val_loss += loss.item()
        val_loss /= len(val_loader)
        
        print(f"====> Epoch [{epoch+1}/{num_epochs}] Validation Loss: {val_loss:.4f} <====")

        scheduler.step(val_loss)


        current_lrs = [group['lr'] for group in optimizer.param_groups]
        if len(current_lrs) == 1:
            print(f"-> Current Learning Rate: {current_lrs[0]}")
        else:
            print(f"-> Current Learning Rates: ViT = {current_lrs[0]} | Decoder = {current_lrs[1]}")
        
        if val_loss < best_val_loss:
            print(f"Validation loss improved ({best_val_loss:.4f} --> {val_loss:.4f}). Saving best model...")
            checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": epoch,
            }
            save_checkpoint(checkpoint, filename="best_vit_transformer.pth.tar")
            best_val_loss = val_loss
            
        print_examples(model, device, train_dataset, num_samples=3)




In [ ]:
train()

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
best_model = ViTtoTransformer(
    embed_size=512,
    vocab_size=len(train_dataset.vocab),
    num_heads=8,
    num_layers=3,
    dropout=0.3,
    train_vit=False
).to(device)

checkpoint = torch.load("/kaggle/input/models/tuanphanthai/vit-transformer/pytorch/default/1/best_vit_transformer.pth.tar")
best_model.load_state_dict(checkpoint["state_dict"])
best_model.eval()
print("Đã tải thành công trọng số của mô hình tốt nhất!")

In [ ]:
results_greedy = evaluate_all_metrics(best_model, test_loader.dataset, device, use_beam_search=False)
results_beam = evaluate_all_metrics(best_model, test_loader.dataset, device, use_beam_search=True, beam_size=3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def autolabel(rects, ax):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), 
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

def plot_and_compare_metrics(dict_greedy, dict_beam):

    metrics = list(dict_greedy.keys())
    
    greedy_scores = [dict_greedy[m] * 100 for m in metrics]
    beam_scores = [dict_beam[m] * 100 for m in metrics]
    
    improvements = [b - g for g, b in zip(greedy_scores, beam_scores)]
    
    df = pd.DataFrame({
        'Metric': metrics,
        'Greedy Search': greedy_scores,
        'Beam Search (k=3)': beam_scores,
        'Cải thiện (+/-)': improvements
    })
    
    df = df.round(2)
    
    print("BẢNG SO SÁNH CHỈ SỐ ĐÁNH GIÁ (Thang điểm 100):")
    display(df) 

    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    rects1 = ax.bar(x - width/2, greedy_scores, width, label='Greedy Search', color='#4C72B0')
    rects2 = ax.bar(x + width/2, beam_scores, width, label='Beam Search (k=3)', color='#55A868')
    
    ax.set_ylabel('Điểm số', fontsize=12)
    ax.set_title('So sánh hiệu suất sinh chú thích: Greedy vs Beam Search', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11)
    ax.legend(fontsize=11)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
            
    autolabel(rects1, ax)
    autolabel(rects2, ax)
    
    fig.tight_layout()
    plt.show()

plot_and_compare_metrics(results_greedy, results_beam)

## 5. Inference Module & Visualization


In [ ]:
def visualize_attention(image_path, model, dataset, device):
    from transformers import ViTImageProcessor
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
    
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize((224, 224), Image.LANCZOS)
    
    test_img = processor(images=img, return_tensors="pt")['pixel_values'].to(device)
    
    model.eval()
    predicted_tokens, alphas = model.caption_image(test_img, dataset.vocab)
    beam_tokens, beam_alphas = model.caption_image_beam_search(test_img, dataset.vocab, beam_size=3)
    
    
    cleaned_tokens = []
    cleaned_alphas = []
    beam_cleaned_tokens = []
    beam_cleaned_alphas = []
    
    for i, token in enumerate(predicted_tokens):
        if token in ["<SOS>", "<PAD>"]:
            continue
        if token == "<EOS>":
            break
        cleaned_tokens.append(token)
        if alphas is not None:
            patch_alphas = alphas[i].view(14, 14).detach().numpy()
            cleaned_alphas.append(patch_alphas)

    for i, token in enumerate(beam_tokens):
        if token in ["<SOS>", "<PAD>"]:
            continue
        if token == "<EOS>":
            break
        beam_cleaned_tokens.append(token)
        if beam_alphas is not None:
            patch_alphas = beam_alphas[i-1].view(14, 14).detach().numpy()
            beam_cleaned_alphas.append(patch_alphas)
            
    print("Generated Caption (GREEDY):", " ".join(cleaned_tokens))
    print("Generated Caption (BEAM K=3):", " ".join(beam_cleaned_tokens))
    

    if len(cleaned_alphas) > 0:
        total_plots = len(cleaned_tokens) + 1 
        num_cols = 4
        num_rows = math.ceil(total_plots / num_cols)
    
        fig, axes = plt.subplots(num_rows, num_cols,
                                 figsize=(num_cols * 4, num_rows * 4))
        axes = axes.flatten()
    
        axes[0].imshow(img_resized)
        axes[0].set_title("Original Image", fontsize=13)
        axes[0].axis("off")
    
        for i, (token, alpha_map) in enumerate(zip(cleaned_tokens, cleaned_alphas)):
            ax = axes[i + 1]
            ax.imshow(img_resized)
            alpha_map_resized = cv2.resize(alpha_map, (224, 224))
            ax.imshow(alpha_map_resized, cmap='jet', alpha=0.5)
            ax.set_title(token, fontsize=13)
            ax.axis("off")
    
        for j in range(total_plots, len(axes)):
            axes[j].axis("off")

    plt.tight_layout()
    plt.show()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
visualize_attention("/kaggle/input/datasets/eeshawn/flickr30k/flickr30k_images/100759042.jpg", best_model, train_dataset, device)
